# 7 — Architecture Comparison on the Final Cleaned Data

Trains **SVM**, **PatchTST** and **InceptionTime** on exactly the dataset the
GRU pipeline ends up with, so every architecture is compared on identical
inputs.

**Input:** `./final_split_data_HybridNorm_BestFeatures_Siamese`
(hybrid normalization -> channel search -> Impostor Triplets cleaning),
the same folder notebooks 5 and 6 consume.

**Output:** `./results/architecture_comparison.txt` -- one row per model per
run, plus a printed summary.

**Runs:** 2 per model, matching notebooks 5 and 6.

The two deep models read the `(N, 288, 3)` tensor directly; the SVM reads
`catch22` descriptors extracted from it, since it cannot take a sequence.

**Run after:** notebook 3.


In [15]:
# ══════════════════════════════════════════════════════════════
# Must run before importing torch/sklearn. Same OpenMP guard as
# notebook 6 -- mixing the MPS backend with scikit-learn's threaded
# neighbour search in one process segfaults on this machine.
# ══════════════════════════════════════════════════════════════
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

## 1. Load the Final Cleaned Data

In [16]:
import pickle
import numpy as np


def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]


folder = "./final_split_data_HybridNorm_BestFeatures_Siamese"

X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")

train_counts = np.bincount(y_train.astype(int), minlength=2)
val_counts   = np.bincount(y_val.astype(int),   minlength=2)
test_counts  = np.bincount(y_test.astype(int),  minlength=2)
class_ratio  = train_counts[0] / train_counts[1]

print(f"Train: {X_train.shape} | NonSEP={train_counts[0]} | SEP={train_counts[1]}")
print(f"Val  : {X_val.shape} | NonSEP={val_counts[0]} | SEP={val_counts[1]}")
print(f"Test : {X_test.shape} | NonSEP={test_counts[0]} | SEP={test_counts[1]}")
print(f"Training class ratio (NonSEP:SEP) : {class_ratio:.1f}:1")

Train: (12380, 288, 3) | NonSEP=12262 | SEP=118
Val  : (1780, 288, 3) | NonSEP=1763 | SEP=17
Test : (3559, 288, 3) | NonSEP=3525 | SEP=34
Training class ratio (NonSEP:SEP) : 103.9:1


## 2. Metrics

Identical `compute_metrics` to notebooks 1--6 so every number in the paper
comes from the same function.

In [17]:
import time
import copy
import warnings
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, brier_score_loss

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    pod       = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    recall    = pod
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0
    hss1 = (TP - FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    far  = FP / (TP + FP) if (TP + FP) > 0 else float("nan")
    bias = (TP + FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    return {
        "TP": int(TP), "TN": int(TN), "FP": int(FP), "FN": int(FN),
        "tss": tss, "far": far, "bias": bias,
        "hss1": hss1, "hss2": hss2, "gss": gss,
        "pod": pod, "recall": recall, "precision": precision,
        "f1": f1, "accuracy": accuracy,
    }

Using device: mps


## 3. PatchTST

Hugging Face `PatchTSTForClassification`. Configuration is the best of the
eight-point grid swept in the companion study, carried over unchanged.
Class-weighted cross-entropy handles the imbalance, matching the weighted
objective used for the GRU.

In [18]:
from transformers import PatchTSTConfig, PatchTSTForClassification

PATCHTST_PARAMS = {
    "patch_length": 12,
    "patch_stride": 12,
    "d_model": 32,
    "num_attention_heads": 4,
    "num_hidden_layers": 1,
    "ffn_dim": 128,
    "dropout": 0.2,
    "lr": 1e-3,
    "batch_size": 128,
}


def build_patchtst_model(params, seq_len, n_channels):
    config = PatchTSTConfig(
        num_input_channels=n_channels,
        context_length=seq_len,
        num_targets=2,
        patch_length=params["patch_length"],
        patch_stride=params["patch_stride"],
        d_model=params["d_model"],
        num_attention_heads=params["num_attention_heads"],
        num_hidden_layers=params["num_hidden_layers"],
        ffn_dim=params["ffn_dim"],
        attention_dropout=params["dropout"],
        positional_dropout=params["dropout"],
        ff_dropout=params["dropout"],
        head_dropout=params["dropout"],
        pooling_type="mean",
        use_cls_token=True,
        norm_type="layernorm",
        channel_attention=True,
        problem_type="single_label_classification",
        scaling=None,
    )
    return PatchTSTForClassification(config)


def train_and_evaluate_patchtst(params, X_train, y_train, X_val, y_val,
                                X_test, y_test, class_ratio,
                                max_epochs=30, patience=3):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.long).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    model = build_patchtst_model(params, X_train.shape[1], X_train.shape[2]).to(device)

    class_weights = torch.tensor([1.0, float(class_ratio)],
                                 dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"],
                                  weight_decay=1e-4)

    batch_size = params["batch_size"]
    n_samples  = X_tr.shape[0]
    best_val_loss, best_state, patience_count = float("inf"), None, 0

    t0 = time.time()
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n_samples, device=device)
        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            logits = model(past_values=X_tr[idx]).prediction_logits
            loss = criterion(logits, y_tr[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(past_values=X_va).prediction_logits,
                                 y_va).item()

        if val_loss < best_val_loss:
            best_val_loss, patience_count = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break
    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()
    with torch.no_grad():
        logits  = model(past_values=X_te).prediction_logits
        y_prob  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        val_log = model(past_values=X_va).prediction_logits
        y_val_prob = torch.softmax(val_log, dim=1)[:, 1].cpu().numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics["brier"]      = brier_score_loss(y_test, y_prob)
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    del model, optimizer, X_tr, y_tr, X_va, y_va, X_te
    if device.type == "mps":
        torch.mps.empty_cache()

    return metrics, y_val_prob, y_prob

## 4. InceptionTime

Two residual Inception blocks, global average pooling, single logit head.
Weighted BCE, the same objective as the GRU.

In [19]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_channels=32, kernel_sizes=(9, 19, 39),
                 bottleneck_channels=32):
        super().__init__()
        if in_channels > 1:
            self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels,
                                        kernel_size=1, bias=False)
            conv_in = bottleneck_channels
        else:
            self.bottleneck = nn.Identity()
            conv_in = in_channels

        self.conv_list = nn.ModuleList([
            nn.Conv1d(conv_in, out_channels, kernel_size=k, padding=k // 2,
                      bias=False) for k in kernel_sizes])
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False))
        self.bn   = nn.BatchNorm1d(out_channels * (len(kernel_sizes) + 1))
        self.relu = nn.ReLU()

    def forward(self, x):
        xb = self.bottleneck(x)
        out = torch.cat([c(xb) for c in self.conv_list] + [self.maxpool_conv(x)],
                        dim=1)
        return self.relu(self.bn(out))


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=32, depth=3, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        modules, current = [], in_channels
        for _ in range(depth):
            modules.append(InceptionModule(current, out_channels))
            current = out_channels * 4
        self.inception_modules = nn.ModuleList(modules)
        if use_residual:
            self.residual = nn.Sequential(
                nn.Conv1d(in_channels, current, kernel_size=1, bias=False),
                nn.BatchNorm1d(current))
            self.relu = nn.ReLU()

    def forward(self, x):
        residual, out = x, x
        for m in self.inception_modules:
            out = m(out)
        if self.use_residual:
            out = self.relu(out + self.residual(residual))
        return out


class InceptionTimeClassifier(nn.Module):
    def __init__(self, input_channels, out_channels=32, depth=6, dropout=0.3):
        super().__init__()
        block_depth = max(1, depth // 2)
        self.block1 = InceptionBlock(input_channels, out_channels, block_depth)
        self.block2 = InceptionBlock(out_channels * 4, out_channels, block_depth)
        self.gap     = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(out_channels * 4, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)               # (batch, channels, timesteps)
        x = self.block2(self.block1(x))
        x = self.dropout(self.gap(x).squeeze(-1))
        return self.fc(x).squeeze()


INCEPTION_PARAMS = {"out_channels": 16, "depth": 6, "dropout": 0.2,
                    "lr": 1e-3, "batch_size": 128}


def train_and_evaluate_inception(params, X_train, y_train, X_val, y_val,
                                 X_test, y_test, class_ratio,
                                 max_epochs=30, patience=3):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    model = InceptionTimeClassifier(X_train.shape[2],
                                    out_channels=params["out_channels"],
                                    depth=params["depth"],
                                    dropout=params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=params["lr"],
                                   weight_decay=1e-4)

    batch_size = params["batch_size"]
    n_samples  = X_tr.shape[0]
    best_val_loss, best_state, patience_count = float("inf"), None, 0

    t0 = time.time()
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n_samples, device=device)
        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            loss = criterion(model(X_tr[idx]), y_tr[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss, patience_count = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break
    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob     = torch.sigmoid(model(X_te)).cpu().numpy()
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics["brier"]      = brier_score_loss(y_test, y_prob)
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    del model, optimizer, X_tr, y_tr, X_va, y_va, X_te
    if device.type == "mps":
        torch.mps.empty_cache()

    return metrics, y_val_prob, y_prob

## 5. SVM

Classical baseline. An SVM cannot consume the raw `(N, 288, 3)` tensor, so each
channel is first summarised by its 22 `catch22` descriptors, giving 66 features
per sample -- the same representation the companion study used for its SVM.
Hyperparameters are the ones that study selected on its hybrid validation
split, carried over unchanged.

Expect the SVM's `raw` row to be degenerate. At $104{:}1$ imbalance the
internally Platt-scaled `predict_proba` never crosses 0.5, so thresholding it
predicts every sample NonSEP and TSS is exactly 0. The `calibrated` row is the
meaningful one, and it is the row the figures read.


In [20]:
from sktime.transformations.panel.catch22 import Catch22
from sklearn.svm import SVC

# RBF kernel, C=0.1: the best of the ten-point SVM grid swept in the
# companion study, selected on its hybrid validation split.
SVM_PARAMS = {
    "kernel": "rbf",
    "C": 0.1,
    "gamma": "scale",
    "class_weight": "balanced",
    "cache_size": 2000,
    "random_state": 0,
}


def extract_catch22(X, label):
    """catch22 expects (samples, channels, timesteps); our tensors are
    (samples, timesteps, channels), so the last two axes are swapped."""
    t0 = time.time()
    F = np.asarray(Catch22(catch24=False).fit_transform(np.swapaxes(X, 1, 2)))
    F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
    print(f"  {label:<5} {X.shape} -> {F.shape}   ({time.time() - t0:.0f}s)")
    return F


# Extracted once and reused by every run; this is the slow part (~5 min).
print("catch22 extraction")
C22_TRAIN = extract_catch22(X_train, "train")
C22_VAL   = extract_catch22(X_val,   "val")
C22_TEST  = extract_catch22(X_test,  "test")


def train_and_evaluate_svm(params, X_train, y_train, X_val, y_val,
                           X_test, y_test, class_ratio):
    """Same contract as the two deep trainers: raw test metrics plus the
    validation and test probabilities that Section 6 calibrates.

    `class_weight="balanced"` plays the role the weighted objective plays
    for the GRU. SVC is deterministic given the data, so both runs return
    identical numbers; both are still recorded to keep the file format
    consistent with the other models.
    """
    model = SVC(probability=True, **params)

    t0 = time.time()
    model.fit(C22_TRAIN, y_train.astype(int))
    train_time = time.time() - t0

    t0 = time.time()
    y_val_prob = model.predict_proba(C22_VAL)[:, 1]
    y_prob     = model.predict_proba(C22_TEST)[:, 1]
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), (y_prob >= 0.5).astype(int))
    metrics["brier"]      = brier_score_loss(y_test, y_prob)
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time
    return metrics, y_val_prob, y_prob


catch22 extraction
  train (12380, 288, 3) -> (12380, 66)   (109s)
  val   (1780, 288, 3) -> (1780, 66)   (15s)
  test  (3559, 288, 3) -> (3559, 66)   (30s)


## 6. Run All Three Models

Two runs each, then class-balanced Platt calibration on the validation split
-- the same convention used everywhere else in this project.

In [21]:
from sklearn.linear_model import LogisticRegression

N_RUNS = 2

MODELS = [
    ("SVM",           SVM_PARAMS,       train_and_evaluate_svm),
    ("PatchTST",      PATCHTST_PARAMS,  train_and_evaluate_patchtst),
    ("InceptionTime", INCEPTION_PARAMS, train_and_evaluate_inception),
]

all_rows = []

for name, params, trainer in MODELS:
    print("═" * 70)
    print(f"  {name}   |   {params}")
    print(f"  Data: {X_train.shape} | ratio {class_ratio:.1f}:1 | {N_RUNS} runs")
    print("═" * 70)

    for run in range(1, N_RUNS + 1):
        metrics, y_val_prob, y_test_prob = trainer(
            copy.deepcopy(params), X_train, y_train, X_val, y_val,
            X_test, y_test, class_ratio)

        all_rows.append({**metrics, "model": name, "run": run, "kind": "raw"})
        print(f"  run {run} raw        : TSS={metrics['tss']:.4f}  "
              f"PoD={metrics['pod']:.4f}  HSS2={metrics['hss2']:.4f}  "
              f"Brier={metrics['brier']:.4f}  ({metrics['train_time']:.0f}s)")

        # class-balanced Platt scaling, fitted on the validation split
        calibrator = LogisticRegression(class_weight="balanced")
        calibrator.fit(y_val_prob.reshape(-1, 1), y_val.astype(int))
        y_test_cal = calibrator.predict_proba(y_test_prob.reshape(-1, 1))[:, 1]

        cal = compute_metrics(y_test.astype(int), (y_test_cal >= 0.5).astype(int))
        cal["brier"] = brier_score_loss(y_test, y_test_cal)
        cal["train_time"] = float("nan")
        cal["infer_time"] = float("nan")
        all_rows.append({**cal, "model": name, "run": run, "kind": "calibrated"})
        print(f"  run {run} calibrated : TSS={cal['tss']:.4f}  "
              f"PoD={cal['pod']:.4f}  HSS2={cal['hss2']:.4f}  "
              f"Brier={cal['brier']:.4f}")
    print()

══════════════════════════════════════════════════════════════════════
  SVM   |   {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000, 'random_state': 0}
  Data: (12380, 288, 3) | ratio 103.9:1 | 2 runs
══════════════════════════════════════════════════════════════════════
  run 1 raw        : TSS=0.0000  PoD=0.0000  HSS2=0.0000  Brier=0.0093  (27s)
  run 1 calibrated : TSS=0.4233  PoD=0.5294  HSS2=0.0681  Brier=0.2117
  run 2 raw        : TSS=0.0000  PoD=0.0000  HSS2=0.0000  Brier=0.0093  (27s)
  run 2 calibrated : TSS=0.4233  PoD=0.5294  HSS2=0.0681  Brier=0.2117

══════════════════════════════════════════════════════════════════════
  PatchTST   |   {'patch_length': 12, 'patch_stride': 12, 'd_model': 32, 'num_attention_heads': 4, 'num_hidden_layers': 1, 'ffn_dim': 128, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 128}
  Data: (12380, 288, 3) | ratio 103.9:1 | 2 runs
══════════════════════════════════════════════════════════════════════
  run 1

## 7. Summary and Save

In [22]:
import numpy as np

print("═" * 78)
print(f"{'Model':<16}{'Kind':<12}{'TSS':>8}{'Brier':>9}{'PoD':>8}"
      f"{'HSS2':>8}{'Prec':>8}{'Acc':>8}")
print("─" * 78)
for name, _, _ in MODELS:
    for kind in ("raw", "calibrated"):
        sel = [r for r in all_rows if r["model"] == name and r["kind"] == kind]
        if not sel:
            continue
        m = {k: float(np.mean([r[k] for r in sel]))
             for k in ("tss", "brier", "pod", "hss2", "precision", "accuracy")}
        print(f"{name:<16}{kind:<12}{m['tss']:>8.4f}{m['brier']:>9.4f}"
              f"{m['pod']:>8.4f}{m['hss2']:>8.4f}{m['precision']:>8.4f}"
              f"{m['accuracy']:>8.4f}")
print("═" * 78)

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)
path = os.path.join(RESULTS_DIR, "architecture_comparison.txt")

header = ("model,run,kind,TP,TN,FP,FN,tss,hss1,hss2,gss,pod,recall,precision,"
          "f1,accuracy,brier,far,bias,train_time,infer_time")
with open(path, "w") as f:
    f.write(header + "\n")
    for r in all_rows:
        f.write(
            f"{r['model']},{r['run']},{r['kind']},"
            f"{r['TP']},{r['TN']},{r['FP']},{r['FN']},"
            f"{r['tss']:.6f},{r['hss1']:.6f},{r['hss2']:.6f},{r['gss']:.6f},"
            f"{r['pod']:.6f},{r['recall']:.6f},{r['precision']:.6f},"
            f"{r['f1']:.6f},{r['accuracy']:.6f},{r['brier']:.6f},"
            f"{r['far']:.6f},{r['bias']:.6f},"
            f"{r.get('train_time', float('nan')):.4f},"
            f"{r.get('infer_time', float('nan')):.4f}\n")

print(f"Saved: {path}")

══════════════════════════════════════════════════════════════════════════════
Model           Kind             TSS    Brier     PoD    HSS2    Prec     Acc
──────────────────────────────────────────────────────────────────────────────
SVM             raw           0.0000   0.0093  0.0000  0.0000  0.0000  0.9904
SVM             calibrated    0.4233   0.2117  0.5294  0.0681  0.0459  0.8904
PatchTST        raw           0.5090   0.0698  0.6029  0.0905  0.0583  0.9032
PatchTST        calibrated    0.5703   0.1406  0.6912  0.0819  0.0529  0.8774
InceptionTime   raw           0.5080   0.0963  0.6176  0.0812  0.0530  0.8877
InceptionTime   calibrated    0.5299   0.1372  0.6765  0.0633  0.0426  0.8518
══════════════════════════════════════════════════════════════════════════════
Saved: ./results/architecture_comparison.txt
